# DOI Registry Resource Type Resolution

This notebook resolves resource type discrepancies (`msp_type` vs `ogc_type`) by querying official **DOI Registries** (**Crossref** and **DataCite**) to determine the author/publisher registered ground-truth resource classification.

### 1. Interactive Single Record Inspector (Step-by-Step Debugger)

Set the `TEST_IDENTIFIER`, `TEST_MSP_TYPE`, and `TEST_OGC_TYPE` variables below to test any individual record. The cell will print every step: DOI extraction, API lookup requests, raw types returned, canonical class mappings, and the final decision.

In [ ]:
import re
import json
import time
import requests
from urllib.parse import unquote

# ==============================================================================
# INTERACTIVE SINGLE RECORD TEST
# (Change these variables to test any identifier/DOI against the registries)
# ==============================================================================
TEST_IDENTIFIER = "10.5281/zenodo.10454359"  # Example DOI / URL / Identifier
TEST_MSP_TYPE   = "Dataset"                          # Type currently in MSP / Solr
TEST_OGC_TYPE   = "Journal Article"                                  # Type currently in OGC catalogue
USER_EMAIL      = "makh@k2x.tech"                             # Email for Crossref polite pool
# ==============================================================================

DOI_RE = re.compile(r"(10\.\d{4,9}/[^\s\"<>?#]+)", re.I)

# Mapping registry types to canonical classes
CROSSREF = {
    "journal-article": "article", "proceedings-article": "conference",
    "book-chapter": "chapter", "book": "book", "monograph": "book",
    "edited-book": "book", "reference-book": "book", "dataset": "dataset",
    "report": "report", "report-component": "report", "dissertation": "thesis",
    "posted-content": "preprint", "other": "other", "peer-review": "other",
    "proceedings": "conference", "component": "other",
}
DATACITE = {
    "journalarticle": "article", "conferencepaper": "conference",
    "conferenceproceeding": "conference", "bookchapter": "chapter",
    "book": "book", "dataset": "dataset", "report": "report",
    "dissertation": "thesis", "preprint": "preprint", "software": "software",
    "text": "text", "audiovisual": "audiovisual", "image": "image",
    "model": "model", "collection": "collection", "event": "event",
    "interactiveresource": "interactive", "computationalnotebook": "notebook",
    "presentation": "presentation", "service": "service", "other": "other",
    "workflow": "software", "physicalobject": "other", "sound": "audiovisual",
}
LABEL = {
    "journal article": "article", "dataset": "dataset", "report": "report",
    "conference proceeding": "conference", "conference object": "conference",
    "book chapter": "chapter", "part of book or chapter of book": "chapter",
    "book": "book", "dissertation": "thesis", "preprint": "preprint",
    "software": "software", "text": "text", "audiovisual": "audiovisual",
    "image": "image", "model": "model", "collection": "collection",
    "event": "event", "interactive resource": "interactive",
    "computational notebook": "notebook", "presentation": "presentation",
    "service": "service", "other": "other", "data paper": "article",
}

def extract_doi(identifier):
    s = unquote(str(identifier))
    if s.startswith("oai:") or "youtu" in s or "vimeo" in s:
        return None
    m = DOI_RE.search(s)
    return m.group(1).rstrip(".,;)").lower() if m else None

def inspect_single_record(identifier, msp_type="journal article", ogc_type="dataset", email="makh@k2x.tech"):
    print("=" * 75)
    print(" STEP-BY-STEP DOI REGISTRY TYPE VALIDATION")
    print("=" * 75)
    print(f"1. INPUT VALUES:")
    print(f"   • Identifier : {identifier}")
    print(f"   • MSP Type   : {msp_type}")
    print(f"   • OGC Type   : {ogc_type}")
    print("-" * 75)
    
    # Step 1: Extract DOI
    doi = extract_doi(identifier)
    print(f"2. DOI EXTRACTION:")
    if doi:
        print(f"   [+] Extracted DOI: {doi}")
    else:
        print(f"   [-] No valid DOI pattern found in identifier (e.g. non-DOI/OAI/web link).")
        print(f"\nFINAL DECISION:")
        print(f"   • correct_side : UNRESOLVED")
        print(f"   • correct_type : (requires manual / web inspection)")
        print("=" * 75)
        return
        
    print("-" * 75)
    print("3. QUERYING OFFICIAL DOI REGISTRIES (Crossref & DataCite):")
    hdr = {"User-Agent": f"msp-type-check/1.0 (mailto:{email})"}
    session = requests.Session()
    
    raw_type = None
    canonical_class = None
    registry_src = "none"
    record_title = "N/A"
    publisher = "N/A"
    
    # Check Crossref first
    crossref_url = f"https://api.crossref.org/works/{doi}"
    print(f"   [A] Querying Crossref: {crossref_url}")
    try:
        r = session.get(crossref_url, params={"mailto": email}, headers=hdr, timeout=20)
        print(f"       -> HTTP Status Code : {r.status_code}")
        if r.status_code == 200:
            msg = r.json().get("message", {})
            raw_type = msg.get("type", "")
            publisher = msg.get("publisher", "Unknown")
            record_title = msg.get("title", [""])[0] if msg.get("title") else "N/A"
            registry_src = "crossref"
            canonical_class = CROSSREF.get(raw_type, "other")
            print(f"       ✓ MATCH FOUND in Crossref!")
            print(f"       • Title          : {record_title}")
            print(f"       • Publisher      : {publisher}")
            print(f"       • Raw Type       : {raw_type}")
            print(f"       • Canonical Class: {canonical_class}")
    except Exception as e:
        print(f"       Crossref request error: {e}")
        
    # If not found in Crossref, query DataCite
    if not raw_type:
        datacite_url = f"https://api.datacite.org/dois/{doi}"
        print(f"   [B] Querying DataCite: {datacite_url}")
        try:
            r = session.get(datacite_url, headers=hdr, timeout=20)
            print(f"       -> HTTP Status Code : {r.status_code}")
            if r.status_code == 200:
                attrs = r.json().get("data", {}).get("attributes", {})
                types_dict = attrs.get("types", {})
                raw_type = types_dict.get("resourceTypeGeneral") or types_dict.get("citeproc") or types_dict.get("resourceType") or ""
                publisher = attrs.get("publisher", "Unknown")
                titles = attrs.get("titles", [{}])
                record_title = titles[0].get("title", "N/A") if titles else "N/A"
                registry_src = "datacite"
                canonical_class = DATACITE.get(str(raw_type).lower(), "other")
                print(f"       ✓ MATCH FOUND in DataCite!")
                print(f"       • Title          : {record_title}")
                print(f"       • Publisher      : {publisher}")
                print(f"       • Raw Type       : {raw_type}")
                print(f"       • Canonical Class: {canonical_class}")
        except Exception as e:
            print(f"       DataCite request error: {e}")
            
    if not raw_type:
        print(f"   [-] DOI was not found in either Crossref or DataCite.")
        
    print("-" * 75)
    print("4. CANONICAL CLASS COMPARISON:")
    msp_class = LABEL.get(str(msp_type).strip().lower(), "unknown")
    ogc_class = LABEL.get(str(ogc_type).strip().lower(), "unknown")
    
    print(f"   • Registry Official Class : {canonical_class} (Source: {registry_src})")
    print(f"   • MSP Type (\"{msp_type}\") -> Class : {msp_class}")
    print(f"   • OGC Type (\"{ogc_type}\") -> Class : {ogc_class}")
    
    msp_ok = (msp_class == canonical_class) and (canonical_class is not None)
    ogc_ok = (ogc_class == canonical_class) and (canonical_class is not None)
    
    print(f"   • MSP matches Registry Class? : {'✓ YES' if msp_ok else '✗ NO'}")
    print(f"   • OGC matches Registry Class? : {'✓ YES' if ogc_ok else '✗ NO'}")
    
    print("-" * 75)
    print("5. FINAL DECISION & OUTPUT:")
    if not canonical_class or registry_src == "none":
        correct_side = "unresolved"
        correct_type = ""
        reason = "DOI could not be resolved in registries."
    elif msp_ok and not ogc_ok:
        correct_side = "msp"
        correct_type = msp_type
        reason = f"MSP type (\"{msp_type}\") matches registry class (\"{canonical_class}\"), while OGC does not."
    elif ogc_ok and not msp_ok:
        correct_side = "ogc"
        correct_type = ogc_type
        reason = f"OGC type (\"{ogc_type}\") matches registry class (\"{canonical_class}\"), while MSP does not."
    elif msp_ok and ogc_ok:
        correct_side = "msp"
        correct_type = msp_type
        reason = f"Both MSP and OGC agree with registry class (\"{canonical_class}\")."
    else:
        correct_side = "neither"
        correct_type = ""
        reason = f"Neither MSP (\"{msp_type}\") nor OGC (\"{ogc_type}\") matches registry class (\"{canonical_class}\")."
        
    print(f"   • correct_side : {correct_side.upper()}")
    print(f"   • correct_type : \"{correct_type}\"")
    print(f"   • Explanation  : {reason}")
    print("=" * 75)

# Execute the diagnostic check
inspect_single_record(
    identifier=TEST_IDENTIFIER,
    msp_type=TEST_MSP_TYPE,
    ogc_type=TEST_OGC_TYPE,
    email=USER_EMAIL
)
# 10.1101/2020.01.22.915835

### 2. Batch DOI Resolution on Mismatched Dataset

Processes all rows in `data/msp_type_mismatches.csv` using multithreading with local cache (`registry_cache.json`), resolving `correct_side` and `correct_type` for the entire corpus.

In [ ]:
import json, os, re, sys, time
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import unquote
import pandas as pd
import requests

# ---------------- settings ----------------
# Auto-detect data folder whether run from repo root or notebooks/
if os.path.exists("data/msp_type_mismatches.csv"):
    DATA_DIR = "data"
elif os.path.exists("../data/msp_type_mismatches.csv"):
    DATA_DIR = "../data"
else:
    DATA_DIR = "."

INFILE     = os.path.join(DATA_DIR, "msp_type_mismatches.csv")
OUTFILE    = os.path.join(DATA_DIR, "msp_type_mismatches_with_correct_type.csv")
CACHE_FILE = os.path.join(DATA_DIR, "registry_cache.json")
EMAIL      = "you@example.org"   # Crossref polite pool
WORKERS    = 4                # Polite concurrency
LIMIT      = None             # None = all rows
# ------------------------------------------


DOI_RE = re.compile(r"(10\.\d{4,9}/[^\s\"<>?#]+)", re.I)

# Mapping registry types to canonical classes
CROSSREF = {
    "journal-article": "article", "proceedings-article": "conference",
    "book-chapter": "chapter", "book": "book", "monograph": "book",
    "edited-book": "book", "reference-book": "book", "dataset": "dataset",
    "report": "report", "report-component": "report", "dissertation": "thesis",
    "posted-content": "preprint", "other": "other", "peer-review": "other",
    "proceedings": "conference", "component": "other",
}
DATACITE = {
    "journalarticle": "article", "conferencepaper": "conference",
    "conferenceproceeding": "conference", "bookchapter": "chapter",
    "book": "book", "dataset": "dataset", "report": "report",
    "dissertation": "thesis", "preprint": "preprint", "software": "software",
    "text": "text", "audiovisual": "audiovisual", "image": "image",
    "model": "model", "collection": "collection", "event": "event",
    "interactiveresource": "interactive", "computationalnotebook": "notebook",
    "presentation": "presentation", "service": "service", "other": "other",
    "workflow": "software", "physicalobject": "other", "sound": "audiovisual",
}
LABEL = {
    "journal article": "article", "dataset": "dataset", "report": "report",
    "conference proceeding": "conference", "conference object": "conference",
    "book chapter": "chapter", "part of book or chapter of book": "chapter",
    "book": "book", "dissertation": "thesis", "preprint": "preprint",
    "software": "software", "text": "text", "audiovisual": "audiovisual",
    "image": "image", "model": "model", "collection": "collection",
    "event": "event", "interactive resource": "interactive",
    "computational notebook": "notebook", "presentation": "presentation",
    "service": "service", "other": "other", "data paper": "article",
}
def extract_doi(identifier):
    s = unquote(str(identifier))
    if s.startswith("oai:") or "youtu" in s or "vimeo" in s:
        return None
    m = DOI_RE.search(s)
    return m.group(1).rstrip(".,;)").lower() if m else None

def lookup(doi, session, email):
    """Query Crossref and DataCite with rate-limiting backoff and retries."""
    hdr = {"User-Agent": f"msp-type-check/1.0 (mailto:{email})"}
    
    # 1. Query Crossref
    for attempt in range(3):
        try:
            r = session.get(f"https://api.crossref.org/works/{doi}",
                            params={"mailto": email}, headers=hdr, timeout=15)
            if r.status_code == 200:
                t = r.json().get("message", {}).get("type", "")
                return t, CROSSREF.get(t, "other"), "crossref"
            elif r.status_code in (429, 503, 504):
                time.sleep(1 + attempt * 2)
                continue
            elif r.status_code == 404:
                break
        except requests.RequestException:
            time.sleep(1)
            continue
            
    # 2. Query DataCite if not found in Crossref
    for attempt in range(3):
        try:
            r = session.get(f"https://api.datacite.org/dois/{doi}", headers=hdr, timeout=15)
            if r.status_code == 200:
                attrs = r.json().get("data", {}).get("attributes", {})
                ty = attrs.get("types", {})
                t = ty.get("resourceTypeGeneral") or ty.get("citeproc") or ""
                return t, DATACITE.get(str(t).lower(), "other"), "datacite"
            elif r.status_code in (429, 503, 504):
                time.sleep(1 + attempt * 2)
                continue
            elif r.status_code == 404:
                break
        except requests.RequestException:
            time.sleep(1)
            continue
            
    return None, None, "none"

def decide(row):
    reg = row["registry_class"]
    if not isinstance(reg, str) or not row["doi"]:
        return pd.Series({"correct_type": "", "correct_side": "unresolved"})
    msp_ok = LABEL.get(str(row["msp_type"]).strip().lower()) == reg
    ogc_ok = LABEL.get(str(row["ogc_type"]).strip().lower()) == reg
    if msp_ok and not ogc_ok:
        return pd.Series({"correct_type": row["msp_type"], "correct_side": "msp"})
    if ogc_ok and not msp_ok:
        return pd.Series({"correct_type": row["ogc_type"], "correct_side": "ogc"})
    if msp_ok and ogc_ok:
        return pd.Series({"correct_type": row["msp_type"], "correct_side": "msp"})
    return pd.Series({"correct_type": "", "correct_side": "neither"})

def main():
    df = pd.read_csv(INFILE, skipinitialspace=True)
    df.columns = df.columns.str.strip()
    for col in df.columns:
        df[col] = df[col].astype(str).str.strip()

    if LIMIT:
        df = df.head(LIMIT).copy()
    df["doi"] = df["identifier"].map(extract_doi)

    cache = json.load(open(CACHE_FILE)) if os.path.exists(CACHE_FILE) else {}
    # Retries items not in cache or previously failed transiently
    todo = sorted(d for d in df["doi"].dropna().unique() if d not in cache or (cache.get(d) and cache[d][2] == "none"))
    print(f"{len(df)} rows | {df['doi'].notna().sum()} with DOI | "
          f"{len(todo)} lookups to do ({len(cache)} cached)")

    sess = requests.Session()
    def work(d):
        time.sleep(0.1)
        return d, lookup(d, sess, EMAIL)

    if todo:
        try:
            with ThreadPoolExecutor(WORKERS) as ex:
                for i, (d, res) in enumerate(ex.map(work, todo), 1):
                    cache[d] = list(res)
                    if i % 200 == 0:
                        json.dump(cache, open(CACHE_FILE, "w"), indent=2)
                        print(f"  {i}/{len(todo)}")
        finally:
            json.dump(cache, open(CACHE_FILE, "w"), indent=2)

    df["registry_type"]  = df["doi"].map(lambda d: (cache.get(d) or [None])[0] if d else None)
    df["registry_class"] = df["doi"].map(lambda d: (cache.get(d) or [None, None])[1] if d else None)
    df["registry_src"]   = df["doi"].map(lambda d: (cache.get(d) or [None, None, "none"])[2] if d else "none")
    
    decisions = df.apply(decide, axis=1)
    df["correct_type"] = decisions["correct_type"]
    df["correct_side"] = decisions["correct_side"]

    df.drop(columns=["doi", "registry_class"]).to_csv(OUTFILE, index=False)
    print("Saved", OUTFILE)
    print(df["correct_side"].value_counts().to_string())

if __name__ == "__main__":
    main()


### 3. CLI Script Version for Automated Runs

The standalone CLI script with argument parsing for running via terminal or automated workflows.

In [ ]:
#!/usr/bin/env python3
"""
Adds  to msp_type_mismatches.csv by querying official DOI registries
(Crossref and DataCite) to determine authoritative resource types and picking
whichever of msp_type / ogc_type agrees with the registry.
"""
import argparse, json, os, re, sys, time
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import unquote
import pandas as pd
import requests

CACHE_FILE = "registry_cache.json"
DOI_RE = re.compile(r"(10\.\d{4,9}/[^\s"<>?#]+)", re.I)

CROSSREF = {
    "journal-article": "article", "proceedings-article": "conference",
    "book-chapter": "chapter", "book": "book", "monograph": "book",
    "edited-book": "book", "reference-book": "book", "dataset": "dataset",
    "report": "report", "report-component": "report", "dissertation": "thesis",
    "posted-content": "preprint", "other": "other", "peer-review": "other",
    "proceedings": "conference", "component": "other",
}
DATACITE = {
    "journalarticle": "article", "conferencepaper": "conference",
    "conferenceproceeding": "conference", "bookchapter": "chapter",
    "book": "book", "dataset": "dataset", "report": "report",
    "dissertation": "thesis", "preprint": "preprint", "software": "software",
    "text": "text", "audiovisual": "audiovisual", "image": "image",
    "model": "model", "collection": "collection", "event": "event",
    "interactiveresource": "interactive", "computationalnotebook": "notebook",
    "presentation": "presentation", "service": "service", "other": "other",
    "workflow": "software", "physicalobject": "other", "sound": "audiovisual",
}
LABEL = {
    "journal article": "article", "dataset": "dataset", "report": "report",
    "conference proceeding": "conference", "conference object": "conference",
    "book chapter": "chapter", "part of book or chapter of book": "chapter",
    "book": "book", "dissertation": "thesis", "preprint": "preprint",
    "software": "software", "text": "text", "audiovisual": "audiovisual",
    "image": "image", "model": "model", "collection": "collection",
    "event": "event", "interactive resource": "interactive",
    "computational notebook": "notebook", "presentation": "presentation",
    "service": "service", "other": "other", "data paper": "article",
}

def extract_doi(identifier):
    s = unquote(str(identifier))
    if s.startswith("oai:") or "youtu" in s or "vimeo" in s:
        return None
    m = DOI_RE.search(s)
    return m.group(1).rstrip(".,;)").lower() if m else None

def lookup(doi, session, email):
    """Query Crossref and DataCite with rate-limiting backoff and retries."""
    hdr = {"User-Agent": f"msp-type-check/1.0 (mailto:{email})"}
    
    # 1. Crossref
    for attempt in range(3):
        try:
            r = session.get(f"https://api.crossref.org/works/{doi}",
                            params={"mailto": email}, headers=hdr, timeout=15)
            if r.status_code == 200:
                t = r.json().get("message", {}).get("type", "")
                return t, CROSSREF.get(t, "other"), "crossref"
            elif r.status_code in (429, 503, 504):
                time.sleep(1 + attempt * 2)
                continue
            elif r.status_code == 404:
                break
        except requests.RequestException:
            time.sleep(1)
            continue
            
    # 2. DataCite
    for attempt in range(3):
        try:
            r = session.get(f"https://api.datacite.org/dois/{doi}", headers=hdr, timeout=15)
            if r.status_code == 200:
                attrs = r.json().get("data", {}).get("attributes", {})
                ty = attrs.get("types", {})
                t = ty.get("resourceTypeGeneral") or ty.get("citeproc") or ""
                return t, DATACITE.get(str(t).lower(), "other"), "datacite"
            elif r.status_code in (429, 503, 504):
                time.sleep(1 + attempt * 2)
                continue
            elif r.status_code == 404:
                break
        except requests.RequestException:
            time.sleep(1)
            continue
            
    return None, None, "none"

def decide(row):
    reg = row["registry_class"]
    if not isinstance(reg, str) or not row["doi"]:
        return pd.Series({"correct_type": "", "correct_side": "unresolved"})
    msp_ok = LABEL.get(str(row["msp_type"]).strip().lower()) == reg
    ogc_ok = LABEL.get(str(row["ogc_type"]).strip().lower()) == reg
    if msp_ok and not ogc_ok:
        return pd.Series({"correct_type": row["msp_type"], "correct_side": "msp"})
    if ogc_ok and not msp_ok:
        return pd.Series({"correct_type": row["ogc_type"], "correct_side": "ogc"})   
    if msp_ok and ogc_ok:
        return pd.Series({"correct_type": row["msp_type"], "correct_side": "msp"}) 
    return pd.Series({"correct_type": "", "correct_side": "neither"})
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("infile"); ap.add_argument("outfile")
    ap.add_argument("--email", required=True, help="used for Crossref polite pool")
    ap.add_argument("--workers", type=int, default=4)
    a = ap.parse_args()

    df = pd.read_csv(a.infile, skipinitialspace=True)
    df.columns = df.columns.str.strip()
    for col in df.columns:
        df[col] = df[col].astype(str).str.strip()

    df["doi"] = df["identifier"].map(extract_doi)
    cache = json.load(open(CACHE_FILE)) if os.path.exists(CACHE_FILE) else {}
    todo = sorted(d for d in df["doi"].dropna().unique() if d not in cache or (cache.get(d) and cache[d][2] == "none"))
    print(f"{df['doi'].notna().sum()} rows with DOI, {len(todo)} lookups to do", file=sys.stderr)

    sess = requests.Session()
    def work(d):
        time.sleep(0.1)
        return d, lookup(d, sess, a.email)

    if todo:
        with ThreadPoolExecutor(a.workers) as ex:
            for i, (d, res) in enumerate(ex.map(work, todo), 1):
                cache[d] = list(res)
                if i % 200 == 0:
                    json.dump(cache, open(CACHE_FILE, "w"), indent=2)
                    print(f"  {i}/{len(todo)}", file=sys.stderr)
        json.dump(cache, open(CACHE_FILE, "w"), indent=2)

    df["registry_type"] = df["doi"].map(lambda d: (cache.get(d) or [None])[0] if d else None)
    df["registry_class"] = df["doi"].map(lambda d: (cache.get(d) or [None, None])[1] if d else None)
    df["registry_src"] = df["doi"].map(lambda d: (cache.get(d) or [None, None, "none"])[2] if d else "none")
    
    decisions = df.apply(decide, axis=1)
    df["correct_type"] = decisions["correct_type"]
    df["correct_side"] = decisions["correct_side"]
    df.drop(columns=["doi", "registry_class"]).to_csv(a.outfile, index=False)
    print(df["correct_side"].value_counts().to_string())

if __name__ == "__main__":
    main()
